# Dev Notebook for One-Tower Modeling

In [1]:
import random
import torch
import io
import pyarrow as pa
import os
import copy
from sacred import Experiment
from PIL import Image
from tqdm import tqdm
import numpy as np
import skimage.io as skio
import matplotlib.pyplot as plt
from refer import REFER

from meter.modules.heads import Pooler

from torch.utils.data import DataLoader
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule

from torch.optim import AdamW

from transformers import AutoConfig
from transformers import ElectraTokenizer, ViltFeatureExtractor
from transformers import AutoProcessor, AutoImageProcessor, AutoTokenizer
from transformers import AutoModel, AutoModelForSequenceClassification

from refcoco_utils import get_bounded_subimage
from refcoco_utils import _loss_names

from meter.transforms import keys_to_transforms
from meter.config import ex
from meter.modules import METERTransformerSS
from meter.datamodules.multitask_datamodule import MTDataModule
from meter.datasets.base_dataset import BaseDataset

In [2]:
refer_root = "/home/claytonfields/nlp/code/data/coco"

In [3]:
def _loss_names(d):
    ret = {
        "itm": 0,
        "mlm": 0,
        "mpp": 0,
        "vqa": 0,
        "vcr": 0,
        "vcr_qar": 0,
        "nlvr2": 0,
        "irtr": 0,
        "contras": 0,
        "snli": 0,
        "ref": 0,
        "mrpc" : 0,
        "rte" : 0,
        'wnli' : 0,
        'sst2' : 0,
        'qqp' : 0,
        'qnli' : 0,
        'mnli' : 0,
        'cola' : 0
    }
    ret.update(d)
    return ret

config = {  
    "exp_name":"finetune_mrpc",
    "seed" : 42,
    # "datasets" : ["coco", "vg", "sbu", "gcc"],
    # "datasets" : ["coco", "vg"],
    "datasets" : ["coco"],
    'loss_names' : _loss_names({"itm": 1, "mlm": 1}),
    "batch_size" : 32,  # this is a desired batch size; pl trainer will accumulate gradients when per step batch is smaller.
    "model_type" : 'one-tower',
    
    # One-Tower Settings
    "random_init_encoder" : False,
    "encoder" : "facebook/deit-tiny-patch16-224",
    'encoder_type' : 'image',
    # Transformer Setting
    # 'vit' : "vit_base_patch32_384",
    'hidden_size' : 192,
    'num_heads' : 12,
    'num_layers' : 12,
    'mlp_ratio' : 4,
    'drop_rate' : 0.1,
    

    # Image setting
    "image_encoder" : "facebook/deit-tiny-patch16-224",
    "random_init_vision_encoder" : False,
    "image_encoder_hidden_size" : 192,
    "image_size" : 224,
    "patch_size" : 16,
    "draw_false_image" : 1,
    "image_only" : False,
    "resolution_before" : 224,
    "train_transform_keys" : ["imagenet"],
    "val_transform_keys" : ["imagenet"],

    # Text Setting
    "text_encoder" : "google/electra-small-discriminator",
    "random_init_text_encoder" : False,
    "text_encoder_hidden_size" : 256,
    "vocab_size" : 30522,
    "whole_word_masking" : False, # note that whole_word_masking does not work for RoBERTa
    "mlm_prob" : 0.15,
    "draw_false_text" : 0,
    "vqav2_label_size" : 3129,
    "max_text_len" : 128,

    # CrossLayer Setting
    "num_cross_layers" : 6,
    "cross_layer_hidden_size" : 256,
    "num_cross_layer_heads" : 4,
    "cross_layer_mlp_ratio" : 4,
    "cross_layer_drop_rate" : 0.1,
    
    # Architecture Setting
    "two_tower" : False,
    "multi_modal_encoder" : 'dandelin/vilt-b32-mlm',
    
    
    
    # Optimizer Setting
    "optim_type" : "adamw",
    "learning_rate" : 5e-5,
    "weight_decay" : 0.0,
    "decay_power" : 1,
    "max_epoch" : 3,
    "max_steps" : 100000,
    "warmup_steps" : 0,
    "end_lr" : 0,
    "lr_mult_head" : 5,  # multiply lr for downstream heads
    "lr_mult_cross_modal" : 5,  # multiply lr for the cross-modal module

    # Encoder Settings
    "freeze_image_encoder" : True,
    "freeze_text_encoder" : False,
    'freeze_cross_modal_layers' : True,
    
    'text_only' : False,
    

    # Downstream Setting
    "get_recall_metric" : False,
    
    'freeze' : True,
    
    # "model_type" : "METER",

    # PL Trainer Setting
    "resume_from" : None,
    "fast_dev_run" : False,
    "val_check_interval" : 1.0,
    "test_only" : False,

    "data_root" : "/home/claytonfields/nlp/code/meter/data/arrow",
    "log_dir" : "result",
    "per_gpu_batchsize" : 32,  # you should define this manually with per_gpu_batch_size:#
    "num_gpus" : 1,
    "num_nodes" : 1,
    'load_path' : '',
    # "load_path" : "/home/claytonfields/nlp/code/meter/result/mlm_itm_seed0_from_/meter_electra_small_deit_tiny_p16_is224_bs288_is1M/checkpoints/epoch=43-step=898039.ckpt",
    # "load_path" : '/home/claytonfields/nlp/code/meter/result/mlm_itm_deit_fr_electra_fr_is224_ps16_bs336_pgbs84_ts100k/checkpoints/epoch=5-step=96215.ckpt',
    "num_workers" : 12,
    "precision" : 32
}


In [4]:
# model = METERTransformerSS(_config)

In [5]:
dm = MTDataModule(config, dist=False)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'ElectraTokenizer'. 
The class this function is called from is 'BertTokenizer'.


In [6]:
dm.prepare_data()
dm.setup('train')

/home/claytonfields/nlp/code/meter/meter/datasets/coco_caption_karpathy_dataset.py:18: FutureWarning: promote has been superseded by mode='default'.
  super().__init__(*args, **kwargs, names=names, text_column_name="caption")


In [7]:
data = dm.train_dataset[1]

In [8]:
text = data['text'][0]
text

'A woman holding a cake with her left hand.'

In [9]:
image = data['image'][0]
image = image.unsqueeze(0)
image

tensor([[[[-1.4158, -1.3130, -1.3644,  ...,  1.8037,  1.7865,  1.7352],
          [-1.3302, -1.3130, -1.3815,  ...,  1.8893,  1.8893,  1.8037],
          [-1.2445, -1.2959, -1.2788,  ...,  1.9578,  1.9578,  1.8379],
          ...,
          [-1.8610, -1.9124, -1.9295,  ...,  1.6838,  1.6667,  1.7180],
          [-1.9295, -1.8610, -1.8953,  ...,  1.6324,  1.7180,  1.7865],
          [-1.9295, -1.8953, -1.8782,  ...,  1.7009,  1.7865,  1.8037]],

         [[-1.3880, -1.2829, -1.3179,  ...,  2.1134,  2.0959,  1.9909],
          [-1.3179, -1.2829, -1.3354,  ...,  2.2010,  2.2010,  2.0434],
          [-1.2129, -1.2829, -1.2654,  ...,  2.2710,  2.2710,  2.0784],
          ...,
          [-1.6155, -1.6856, -1.6856,  ...,  1.5007,  1.5707,  1.6583],
          [-1.6856, -1.6155, -1.7031,  ...,  1.5007,  1.6758,  1.7633],
          [-1.7031, -1.6331, -1.7206,  ...,  1.6057,  1.7808,  1.7983]],

         [[-1.2641, -1.1944, -1.2119,  ...,  2.1694,  2.1171,  2.0300],
          [-1.1944, -1.1770, -

### Vit

In [10]:
from transformers.models.vit.modeling_vit import ViTEmbeddings
from transformers import AutoConfig

In [11]:
vit = AutoModel.from_pretrained("google/vit-base-patch16-224")
vit_config = AutoConfig.from_pretrained(config["image_encoder"])
vit_embeds = ViTEmbeddings(vit_config)

Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['vit.pooler.dense.bias', 'vit.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
vit_embeds(image).shape

torch.Size([1, 197, 192])

### MobileVit

In [13]:
mobilevit = AutoModel.from_pretrained("apple/mobilevit-small")
mobilevit

MobileViTModel(
  (conv_stem): MobileViTConvLayer(
    (convolution): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (normalization): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (activation): SiLU()
  )
  (encoder): MobileViTEncoder(
    (layer): ModuleList(
      (0): MobileViTMobileNetLayer(
        (layer): ModuleList(
          (0): MobileViTInvertedResidual(
            (expand_1x1): MobileViTConvLayer(
              (convolution): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
              (normalization): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (activation): SiLU()
            )
            (conv_3x3): MobileViTConvLayer(
              (convolution): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
              (normalization): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_st

In [14]:
mobilevit_embeds = mobilevit.conv_stem(image)
mobilevit_embeds.shape

torch.Size([1, 16, 112, 112])

In [15]:
mobilevit.encoder(mobilevit_embeds)[0].shape

torch.Size([1, 160, 7, 7])

In [16]:
mobilevit.config.model_type

'mobilevit'

### DinoV2

In [17]:
# dino_processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
dino = AutoModel.from_pretrained('facebook/dinov2-base')

In [18]:
dino_embeds = dino.embeddings(image)
dino_embeds.shape

torch.Size([1, 257, 768])

In [19]:
dino.config.hidden_size

768

In [20]:
dino.config.image_size

518

### Convnext

In [45]:
swin =  AutoModel.from_pretrained('microsoft/swin-tiny-patch4-window7-224')

In [50]:
swin_embeds = swin.embeddings(image)
swin_embeds[0].shape

torch.Size([1, 3136, 96])

In [48]:
swin_embeds[1]

(56, 56)

In [49]:
swin.config

SwinConfig {
  "_name_or_path": "microsoft/swin-tiny-patch4-window7-224",
  "architectures": [
    "SwinForImageClassification"
  ],
  "attention_probs_dropout_prob": 0.0,
  "depths": [
    2,
    2,
    6,
    2
  ],
  "drop_path_rate": 0.1,
  "embed_dim": 96,
  "encoder_stride": 32,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "id2label": {
    "0": "tench, Tinca tinca",
    "1": "goldfish, Carassius auratus",
    "2": "great white shark, white shark, man-eater, man-eating shark, Carcharodon carcharias",
    "3": "tiger shark, Galeocerdo cuvieri",
    "4": "hammerhead, hammerhead shark",
    "5": "electric ray, crampfish, numbfish, torpedo",
    "6": "stingray",
    "7": "cock",
    "8": "hen",
    "9": "ostrich, Struthio camelus",
    "10": "brambling, Fringilla montifringilla",
    "11": "goldfinch, Carduelis carduelis",
    "12": "house finch, linnet, Carpodacus mexicanus",
    "13": "junco, snowbird",
    "14": "indigo bunting, indigo finch, indig

In [55]:
swin_embeds[0].shape[1]

3136

In [51]:
swin.encoder(swin_embeds)

TypeError: SwinEncoder.forward() missing 1 required positional argument: 'input_dimensions'

In [53]:
mod_list = []
for name, module in model.named_modules():
    print(name)
    # print(module)
    mod_list.append(module)
mod_list


embeddings
embeddings.patch_embeddings
embeddings.patch_embeddings.projection
embeddings.dropout
encoder
encoder.layer
encoder.layer.0
encoder.layer.0.attention
encoder.layer.0.attention.attention
encoder.layer.0.attention.attention.query
encoder.layer.0.attention.attention.key
encoder.layer.0.attention.attention.value
encoder.layer.0.attention.attention.dropout
encoder.layer.0.attention.output
encoder.layer.0.attention.output.dense
encoder.layer.0.attention.output.dropout
encoder.layer.0.intermediate
encoder.layer.0.intermediate.dense
encoder.layer.0.intermediate.intermediate_act_fn
encoder.layer.0.output
encoder.layer.0.output.dense
encoder.layer.0.output.dropout
encoder.layer.0.layernorm_before
encoder.layer.0.layernorm_after
encoder.layer.1
encoder.layer.1.attention
encoder.layer.1.attention.attention
encoder.layer.1.attention.attention.query
encoder.layer.1.attention.attention.key
encoder.layer.1.attention.attention.value
encoder.layer.1.attention.attention.dropout
encoder.layer.

[ViTModel(
   (embeddings): ViTEmbeddings(
     (patch_embeddings): ViTPatchEmbeddings(
       (projection): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
     )
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (encoder): ViTEncoder(
     (layer): ModuleList(
       (0-11): 12 x ViTLayer(
         (attention): ViTAttention(
           (attention): ViTSelfAttention(
             (query): Linear(in_features=192, out_features=192, bias=True)
             (key): Linear(in_features=192, out_features=192, bias=True)
             (value): Linear(in_features=192, out_features=192, bias=True)
             (dropout): Dropout(p=0.0, inplace=False)
           )
           (output): ViTSelfOutput(
             (dense): Linear(in_features=192, out_features=192, bias=True)
             (dropout): Dropout(p=0.0, inplace=False)
           )
         )
         (intermediate): ViTIntermediate(
           (dense): Linear(in_features=192, out_features=768, bias=True)
           (intermediate_

In [50]:
mod_list[6]

ModuleList(
  (0-11): 12 x ViTLayer(
    (attention): ViTAttention(
      (attention): ViTSelfAttention(
        (query): Linear(in_features=192, out_features=192, bias=True)
        (key): Linear(in_features=192, out_features=192, bias=True)
        (value): Linear(in_features=192, out_features=192, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (output): ViTSelfOutput(
        (dense): Linear(in_features=192, out_features=192, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (intermediate): ViTIntermediate(
      (dense): Linear(in_features=192, out_features=768, bias=True)
      (intermediate_act_fn): GELUActivation()
    )
    (output): ViTOutput(
      (dense): Linear(in_features=768, out_features=192, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (layernorm_before): LayerNorm((192,), eps=1e-12, elementwise_affine=True)
    (layernorm_after): LayerNorm((192,), eps=1e-12, elementwise_affine=True)
  )
)

In [54]:
image_embeds

tensor([[[-2.2259, -1.6213, -1.7132,  ...,  1.6690, -0.1892,  2.8927],
         [ 0.1985,  0.2933, -0.6428,  ...,  0.4650, -1.4210,  1.0205],
         [ 0.5994, -2.0985,  0.5046,  ...,  0.2008, -1.6631,  1.4113],
         ...,
         [ 1.5403,  1.2748,  0.3832,  ..., -0.1644,  1.3798, -0.1337],
         [-0.0650,  0.9501, -0.2193,  ..., -2.9251,  0.5112, -1.0711],
         [-0.6074,  1.2157,  0.5721,  ..., -0.9924,  0.1639,  0.7663]]],
       grad_fn=<AddBackward0>)

In [56]:
x = model.encoder(image_embeds)

In [61]:
batch

{'false_image_0': [tensor([[[[-1.0904, -1.7240, -1.1418,  ..., -1.3987, -1.3987, -1.3815],
            [-1.4500, -1.3302, -1.0904,  ..., -1.2103, -1.1589, -1.0904],
            [-1.5699, -0.8507, -0.8849,  ..., -0.7993, -0.8849, -0.7822],
            ...,
            [ 0.4166,  0.3994,  0.3481,  ..., -1.6213, -1.6213, -1.6213],
            [ 0.4679,  0.4679,  0.4679,  ..., -1.5870, -1.5699, -1.6042],
            [ 0.4851,  0.4508,  0.5022,  ..., -1.7412, -1.7069, -1.7069]],
  
           [[-1.0378, -1.7206, -1.2829,  ..., -1.5455, -1.4755, -1.3880],
            [-1.4755, -1.4405, -1.1779,  ..., -1.4055, -1.3880, -1.2829],
            [-1.6155, -1.0728, -1.0028,  ..., -1.0903, -1.1429, -1.0903],
            ...,
            [ 0.4328,  0.3978,  0.4153,  ..., -1.4930, -1.5280, -1.5280],
            [ 0.3978,  0.3803,  0.4678,  ..., -1.4930, -1.4580, -1.4755],
            [ 0.5028,  0.4153,  0.4153,  ..., -1.6856, -1.6856, -1.6506]],
  
           [[-1.3687, -1.6824, -1.4384,  ..., -1.6127

In [63]:
pl.seed_everything(config["seed"])
model = METERTransformerSS(config)

Seed set to 42
Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-tiny-patch16-224 and are newly initialized: ['vit.pooler.dense.bias', 'vit.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [73]:
batch = next(iter(dm.train_dataloader()))

image_token_type_idx=1
if f"image_{image_token_type_idx - 1}" in batch:
    imgkey = f"image_{image_token_type_idx - 1}"
else:
    imgkey = "image"
    
do_mlm = '_mlm'
text_ids = batch[f"text_ids{do_mlm}"]
text_labels = batch[f"text_labels{do_mlm}"]
text_masks = batch[f"text_masks"]
text_embeds = model.text_embeddings(text_ids)

# if image_embeds is None and image_masks is None:
#     img = batch[imgkey][0]
#     (
#         image_embeds,
#         image_masks,
#         patch_index,
#         image_labels,
#     ) = model.transformer.visual_embed(
#         img,
#         max_image_len=model.hparams.config["max_image_len"],
#         mask_it=mask_image,
#     )
# else:
#     patch_index, image_labels = (
#         None,
#         None,
#     )

image_embeds = model.image_embeddings(batch['image'][0])
image_masks = torch.ones(image_embeds.shape[0],image_embeds.shape[1], dtype=torch.int8)

text_embeds, image_embeds = (
    text_embeds + model.token_type_embeddings(torch.zeros_like(text_masks)),
    image_embeds
    + model.token_type_embeddings(
        torch.full((image_embeds.shape[0],image_embeds.shape[1]), 
        image_token_type_idx)),
)

co_embeds = torch.cat([text_embeds, image_embeds], dim=1)
co_masks = torch.cat([text_masks, image_masks], dim=1)

x = co_embeds

# for i, blk in enumerate(model.encoder.blocks):
#     x, _attn = blk(x, mask=co_masks)

# x = model.transformer.norm(x)
x = model.encoder.encoder(x)[0]
text_feats, image_feats = (
    x[:, : text_embeds.shape[1]],
    x[:, text_embeds.shape[1] :],
)
cls_feats = model.pooler(x)
cls_feats

tensor([[ 0.3106,  0.0976, -0.2814,  ...,  0.0685,  0.0019,  0.2637],
        [ 0.3784, -0.0424, -0.0849,  ...,  0.0863, -0.2325,  0.0848],
        [ 0.4357, -0.0347, -0.2194,  ...,  0.1181, -0.1374,  0.0971],
        ...,
        [ 0.4359,  0.0521, -0.1508,  ...,  0.2696, -0.1454,  0.0362],
        [ 0.4326,  0.0316, -0.3035,  ...,  0.1494,  0.0067,  0.0522],
        [ 0.4394, -0.0619, -0.0889,  ...,  0.1900, -0.0868,  0.0470]],
       grad_fn=<TanhBackward0>)

In [72]:
x[0][:, : text_embeds.shape[1]]

tensor([[[-0.8201, -0.1205,  1.5231,  ...,  2.0439,  0.7495,  0.4758],
         [-1.5890, -0.7528,  1.8772,  ...,  1.3333,  0.7532,  0.8203],
         [-0.0963, -0.3817,  1.3417,  ...,  1.8607,  0.8482,  0.5045],
         ...,
         [-0.4604, -0.9361,  1.4889,  ...,  2.5879,  0.6056,  0.9026],
         [ 0.7614, -0.8684,  0.5383,  ...,  2.7088,  0.3912,  0.5078],
         [-0.8655, -1.3491,  1.1061,  ...,  2.1463,  1.0815,  0.5344]],

        [[-1.3292, -0.7964,  1.8713,  ...,  1.8296,  0.8854,  0.2857],
         [-1.1364, -1.0713,  1.2857,  ...,  2.0159,  1.4823,  0.1607],
         [ 0.3353, -0.4545,  1.4363,  ...,  2.3506,  0.6723,  0.3338],
         ...,
         [-0.9449, -0.9178,  1.4289,  ...,  1.5303,  1.0483,  1.0843],
         [ 0.2386, -0.6453,  1.3253,  ...,  1.8245, -0.2479,  1.1420],
         [-0.7807, -1.4363,  0.9401,  ...,  1.6364,  0.8925,  0.6571]],

        [[-0.5886,  0.4687,  2.4832,  ...,  2.2809,  1.1255,  0.8095],
         [-1.2836, -1.2455,  2.0471,  ...,  2

In [ ]:
data_root = '/home/claytonfields/nlp/code/data/coco'  # contains refclef, refcoco, refcoco+, refcocog and images
dataset = 'refcoco' 
splitBy = 'unc'
refer = REFER(data_root, dataset, splitBy)

In [ ]:
# file_name = 'COCO_train2014_000000173056_1.jpg'
img_id = 98304
img = refer.Imgs[img_id]
I = skio.imread(os.path.join(refer.IMAGE_DIR, img['file_name']))
I

In [ ]:
# from transformers import SwinModel
# swin = SwinModel.from_pretrained("microsoft/swin-tiny-patch4-window7-224")
# swin(image.unsqueeze(0))

In [ ]:
# inputs = preprocessor(images=I, return_tensors="pt")
# inputs

In [ ]:
# preprocessor = AutoImageProcessor.from_pretrained("google/mobilenet_v2_1.4_224")
# model = AutoModel.from_pretrained("google/mobilenet_v2_1.4_224")

# inputs = preprocessor(images=I, return_tensors="pt")

# outputs = model(image)
# outputs

In [ ]:
vit_embeds(image, use_mask_token = True)

### Classification Head

In [ ]:
config = copy.deepcopy(_config)
pl.seed_everything(_config["seed"])
model = METERTransformerSS(config)
model.current_tasks = ['mrpc']


dm = GlueDataModule(_config, 'mrpc', 32)

pl.seed_everything(_config["seed"])

exp_name = f'{_config["exp_name"]}'

os.makedirs(_config["log_dir"], exist_ok=True)
checkpoint_callback = pl.callbacks.ModelCheckpoint(
    save_top_k=1,
    verbose=True,
    monitor="val/the_metric",
    mode="max",
    save_last=True,
)
logger = pl.loggers.TensorBoardLogger(
    _config["log_dir"],
    name=f'{exp_name}_seed{_config["seed"]}_from_{_config["load_path"].split("/")[-1][:-5]}',
)

lr_callback = pl.callbacks.LearningRateMonitor(logging_interval="step")
callbacks = [checkpoint_callback, lr_callback]

num_gpus = (
    _config["num_gpus"]
    if isinstance(_config["num_gpus"], int)
    else len(_config["num_gpus"])
)

grad_steps = max(_config["batch_size"] // (
    _config["per_gpu_batchsize"] * num_gpus * _config["num_nodes"]
), 1)

max_steps = _config["max_steps"] if _config["max_steps"] is not None else None

trainer = pl.Trainer(
    devices=num_gpus,
    num_nodes=_config["num_nodes"],
    precision=_config["precision"],
    # accelerator="ddp",
    benchmark=True,
    deterministic=True,
    max_epochs=_config["max_epoch"] if max_steps is None else 1000,
    max_steps=max_steps,
    callbacks=callbacks,
    logger=logger,
    #prepare_data_per_node=False,
    #replace_sampler_ddp=False,
    accumulate_grad_batches=grad_steps,
    log_every_n_steps=10,
    # flush_logs_every_n_steps=10,
#     resume_from_checkpoint=_config["resume_from"],
    # weights_summary="top",
    fast_dev_run=_config["fast_dev_run"],
    val_check_interval=_config["val_check_interval"],
)

# log_dir = logger.log_dir
# eval_file = 'eval.txt'
# eval_path = os.path.join(log_dir, eval_file )
# setattr(model, f"eval_path", eval_path)
# f = open(eval_path,'w') 
# f.close()

if not _config["test_only"]:
    trainer.fit(model, datamodule=dm)
else:
    trainer.test(model, datamodule=dm)


## Import Calssification Head?

In [ ]:
model_type = model.text_encoder.base_model_prefix
model_name = model_type.capitalize()
model_type = 'bert'
model_name = model_type.capitalize()

In [ ]:
exec_string = f'from transformers.models.{model_type}.modeling_{model_type} import {model_name}ClassificationHead'

In [ ]:
from transformers.models.electra.modeling_electra import ElectraClassificationHead

In [ ]:
exec(exec_string)

## Image Encoder

In [ ]:
model.image_encoder(